In [2]:
#%pip install pandas matplotlib seaborn sqlalchemy ipython-sql==0.4.1 prettytable==2.5.0
#%pip install tabulate  
#%pip install plotly
#%pip install nbformat>=4.2.0

# VDP Analysis
This notebook will be working with the 'VDP_dataset.csv' file, which was extracted from the Dune query at dune.com/queries/7717567. The table will have the following Columns:
 
| Column | Description |
|---|---|
| id | |
| name | |
| website | |
| auth_address | |
| current_commission | |
| total_stake | |
| vdp_stake | |
| Organic_stake | |
| initial_tier | |
| current_tier | |
| dependency_ratio | |
| Rewards_USD | |
| MonthlyReturns_USD | |
| EstimatedMonthlyReturns_USD | |
| underwater_status | |
| graduation_status | |

In [3]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('sqlite:///vdpdata.db')

csv_file = 'Vdp_dataset.csv'
table_name = 'vdp'
df = pd.read_csv(csv_file)
df.to_sql(table_name, engine, if_exists='replace', index=False)
print('successful')

successful


- How many validators received a delegation?
- Which delegation tier did each validator receive? 
- How concentrated is VDP stake among validators?


In [29]:
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px

MONAD = {
    "purple":    "#836EF9",
    "blue":      "#200052",
    "berry":     "#A0055D",
    "off_white": "#FBFAF9",
    "black":     "#0E100F",
}
CORTEX_GRAPHITE = "#5B6470"
CORTEX_SLATE    = "#9BA4B0"

COLOR_GROWTH     = MONAD["purple"]
COLOR_BORDERLINE = CORTEX_SLATE
COLOR_SHAME      = MONAD["berry"]
COLOR_NEUTRAL    = CORTEX_GRAPHITE
COLOR_BG         = MONAD["off_white"]
COLOR_TEXT       = MONAD["black"]

CATEGORY_COLORS = {
    "Growing Ecosystem (>=10%)": COLOR_GROWTH,
    "Borderline (5-10%)": COLOR_BORDERLINE,
    "Not Growing (<5%)": COLOR_SHAME,
    "No Data": CORTEX_SLATE,
}

FONT_FAMILY = "Inter, -apple-system, Helvetica Neue, Arial, sans-serif"  
tufte_template = go.layout.Template()
tufte_template.layout = go.Layout(
    font=dict(family=FONT_FAMILY, size=13, color=COLOR_TEXT),
    paper_bgcolor=COLOR_BG,
    plot_bgcolor=COLOR_BG,
    title=dict(font=dict(size=17, color=MONAD["blue"]), x=0.02, xanchor="left"),
    xaxis=dict(showgrid=False, zeroline=False, showline=True,
               linecolor=CORTEX_SLATE, linewidth=1, ticks="outside", tickcolor=CORTEX_SLATE),
    yaxis=dict(showgrid=True, gridcolor="#EDEBF7", gridwidth=1, zeroline=False,
               showline=False, ticks=""),
    legend=dict(bgcolor="rgba(0,0,0,0)", bordercolor="rgba(0,0,0,0)"),
    margin=dict(l=60, r=30, t=60, b=50),
    colorway=[MONAD["purple"], CORTEX_GRAPHITE, MONAD["berry"], CORTEX_SLATE, MONAD["blue"]],
)
pio.templates["tufte_monad"] = tufte_template
pio.templates.default = "tufte_monad"


def categorize_share(share):
    """
    Classifies a validator's organic-stake share of total stake.
    Threshold rationale: a validator needs organic stake to represent a
    meaningful piece of its own total stake to count as having grown the
    ecosystem, rather than sitting on parked/idle VDP capital.
    """
    if pd.isna(share):
        return "No Data"
    if share >= 0.10:
        return "Growing Ecosystem (>=10%)"
    elif share >= 0.05:
        return "Borderline (5-10%)"
    else:
        return "Not Growing (<5%)"


In [30]:
query = """SELECT COUNT(DISTINCT id) as "Total VDP Validators"
FROM 'vdp'
WHERE initial_tier IS NOT NULL"""
df = pd.read_sql(query,engine)
total = df["Total VDP Validators"].iloc[0]
fig = go.Figure(
    go.Indicator(
        mode="number",
        value=total,
        number={
            "font": {
            "color": MONAD["purple"],
            "family": FONT_FAMILY
        }},
        title={"text": "Total VDP Validators",
        "font": {"color": MONAD["blue"]}}
        ))
fig.update_layout(paper_bgcolor=COLOR_BG)
fig.show()

In [31]:
query = """SELECT initial_tier, COUNT(id) as total_delegators
FROM 'vdp'
WHERE initial_tier IS NOT NULL
GROUP BY initial_tier
ORDER BY count(id) desc"""
df = pd.read_sql(query, engine)

# Tufte principle: with more than two categories, an ordered bar is read
# faster and more accurately than pie-slice angles.
df = df.sort_values("total_delegators")
fig = px.bar(
    df, x="total_delegators", y="initial_tier", orientation="h",
    text="total_delegators",
    title="Initial VDP Tier Distribution",
)
fig.update_traces(marker_color=MONAD["purple"], textposition="outside")
fig.update_layout(xaxis_title="Number of Validators", yaxis_title="", showlegend=False)
fig.show()

In [6]:
import plotly.express as px

query = """SELECT current_tier, COUNT(id) as total_delegators
FROM 'vdp'
WHERE current_tier IS NOT NULL
GROUP BY current_tier
ORDER BY count(id) desc"""
df = pd.read_sql(query, engine)
label = df["current_tier"]
size = df["total_delegators"]
fig = px.pie(df, values=size, names=label, title="VDP Distribution Chart", hole=0.0)
fig.show()

In [7]:
query = """SELECT initial_tier, count(id) as total_delegators
FROM 'vdp'
WHERE initial_tier IS NOT NULL
GROUP BY initial_tier"""
df = pd.read_sql(query, engine)
fig = px.bar(df, x="initial_tier", y="total_delegators", title="VDP Distribution Chart")
#fig.update_layout(, showlegend=False, xaxis_title="Delegation Tier", yaxis="Number of Validators")
fig.show()

Herfindal-Hirschman Index (HHI)

In [8]:
query ="""with shares AS (SELECT vdp_stake * 1.0 / SUM(vdp_stake) OVER() AS share
FROM vdp)
select sum(power(share, 2)) * 10000 AS hhi_10000
FROM shares"""
df = pd.read_sql(query, engine)
df

,hhi_10000
0,54.390096


In [9]:
query="""SELECT dependency_ratio
FROM 'vdp'
"""
df = pd.read_sql(query,engine)
ratio = df["dependency_ratio"]
fig = px.histogram(df, x="dependency_ratio", nbins=5)
fig.show()

In [10]:
query ="""SELECT name, vdp_stake, organic_stake
FROM 'vdp'
"""
df = pd.read_sql(query, engine)
fig = px.scatter(df, y="vdp_stake", x="organic_stake", hover_name="name")
fig.show()

In [11]:
query = """SELECT name As "Validator Name", organic_stake As "Organic Stake"
FROM 'vdp'
ORDER BY organic_stake DESC
LIMIT 20"""

df = pd.read_sql(query,engine)
fig = px.bar(df, x="Validator Name", y="Organic Stake", title="Top 20 Validators by Organic Stake")
fig.show()

In [12]:
query = """SELECT name As "Validator Name", total_stake As "Total Stake"
FROM 'vdp'
ORDER BY total_stake DESC
LIMIT 20"""

df = pd.read_sql(query,engine)
fig = px.bar(df, x="Validator Name", y="Total Stake", title="Top 20 Validators by Total Stake")
fig.show()

In [13]:
query = """
SELECT count(*) as "Total Validators", CASE WHEN graduation_status IS false THEN "undergraduated" else "graduated" 
end as "Status"
from 'vdp'
group by 2
"""
df = pd.read_sql(query,engine)
names = df["Status"]
values = df["Total Validators"]
fig = px.pie(df, names=names, values=values, title="Validator Graduation Status")
fig.show()

In [14]:
query = """
SELECT count(*) as "Total Validators", CASE WHEN underwater_status IS true THEN "Underwater" else "Above Water" 
end as "Status"
from 'vdp'
group by 2
"""
df = pd.read_sql(query,engine)
names = df["Status"]
values = df["Total Validators"]
fig = px.pie(df, names=names, values=values, title="Validator Sustainability")
fig.show()

In [15]:
query ="""SELECT name AS Validator, ROUND(MonthlyReturns_USD,2) AS "Current Monthly Returns", rOUND(EstimatedMonthlyReturns_USD,2) as "Estimated Returns Without VDP"
FROM 'vdp'
"""
df = pd.read_sql(query, engine)
fig = px.scatter(df, x="Current Monthly Returns", y="Estimated Returns Without VDP", hover_name="Validator")
fig.update_layout(yaxis_tickprefix = '$',
xaxis_tickprefix='$')
fig.show()

In [16]:
query = """
SELECT sum(organic_stake) as stake, 'organic' as type
from 'vdp'
union
select sum(vdp_stake) as stake, 'vdp' as type
from 'vdp'
"""
df = pd.read_sql(query,engine)
names = df["type"]
values = df["stake"].astype(int)
fig = px.pie(df, names=names, values=values, title="Stake by Type")
fig.update_traces(hovertemplate="<b>%{label}</b> <br>" + "Stake:%{value:,.0f} MON<br>"
+ "Share: %{percent}<extra></extra>")
fig.show()


In [17]:
query = """
WITH main AS (
    SELECT row_number() over(order by total_stake desc) as row, name, total_stake, organic_stake
    FROM vdp
    ORDER BY total_stake DESC
    LIMIT 50
),

T20 AS (
    SELECT SUM(total_stake) AS top_20_total, SUM(organic_stake) AS top_20_organic
    FROM main
    WHERE row <= 20
),

T10 AS (
    SELECT SUM(total_stake) AS top_10_total, SUM(organic_stake) AS top_10_organic
    FROM main
    WHERE row <= 10
),

T50 AS (
    SELECT SUM(total_stake) AS top_50_total, SUM(organic_stake) AS top_50_organic
    FROM main
    WHERE row <= 50
), 

totals AS (
    SELECT sum(total_stake) AS total, sum(organic_stake) AS organic
    FROM vdp
)

SELECT 'Top 10 Share' AS metric, 
       top_10_total / total AS Current, 
       top_10_organic / organic AS "Without VDP"
FROM totals, T10

UNION ALL

SELECT 'Top 20 Share' AS metric, 
       top_20_total / total AS Current, 
       top_20_organic / organic AS "Without VDP"
FROM totals, T20

UNION ALL

SELECT 'Top 50 Share' AS metric, 
       top_50_total / total AS Current, 
       top_50_organic / organic AS "Without VDP"
FROM totals, T50
"""

df = pd.read_sql(query, engine)
df['Current'] = df['Current'].map('{:.2%}'.format)
df['Without VDP'] = df['Without VDP'].map('{:.2%}'.format)

fig = go.Figure(
    data = [go.Table(
        header=dict(values=list(df.columns),
        align='left'),
        cells=dict(values=[df[col] for col in df.columns],
        align='left')
    )]
)
fig.show()

In [18]:
query = """
WITH main AS (
    SELECT row_number() over(order by total_stake desc) as row, name, total_stake, organic_stake
    FROM vdp
    ORDER BY total_stake DESC
    LIMIT 50
),

T20 AS (
    SELECT SUM(total_stake) AS top_20_total, SUM(organic_stake) AS top_20_organic
    FROM main
    WHERE row <= 21
    and lower(name) is not 'backpack'
),

T10 AS (
    SELECT SUM(total_stake) AS top_10_total, SUM(organic_stake) AS top_10_organic
    FROM main
    WHERE row <= 11
    and lower(name) is not 'backpack'
),

T50 AS (
    SELECT SUM(total_stake) AS top_50_total, SUM(organic_stake) AS top_50_organic
    FROM main
    WHERE row <= 51
    and lower(name) is not 'backpack'
), 

totals AS (
    SELECT sum(total_stake) AS total, sum(organic_stake) AS organic
    FROM vdp
    where lower(name) is not 'backpack'
)

SELECT 'Top 10 Share' AS metric, 
       top_10_total / total AS Current, 
       top_10_organic / organic AS "Without VDP"
FROM totals, T10

UNION ALL

SELECT 'Top 20 Share' AS metric, 
       top_20_total / total AS Current, 
       top_20_organic / organic AS "Without VDP"
FROM totals, T20

UNION ALL

SELECT 'Top 50 Share' AS metric, 
       top_50_total / total AS Current, 
       top_50_organic / organic AS "Without VDP"
FROM totals, T50
"""

df = pd.read_sql(query, engine)
df['Current'] = df['Current'].map('{:.2%}'.format)
df['Without VDP'] = df['Without VDP'].map('{:.2%}'.format)

fig = go.Figure(
    data = [go.Table(
        header=dict(values=list(df.columns),
        align='left'),
        cells=dict(values=[df[col] for col in df.columns],
        align='left')
    )]
)
fig.show()

In [19]:
query = """WITH tot AS (select sum(total_stake) as total, sum(organic_stake) as organic
from 'vdp')

SELECT total_stake/total as "Total Backpack Concentration", organic_stake/organic as "Total Organic Concentration"
from 'vdp', 'tot' 
where lower(name) is 'backpack'

"""

df = pd.read_sql(query, engine)
df['Total Backpack Concentration'] = df['Total Backpack Concentration'].map('{:.2%}'.format)
df['Total Organic Concentration'] = df['Total Organic Concentration'].map('{:.2%}'.format)

fig = go.Figure(
    data = [go.Table(
        header=dict(values=list(df.columns),
        align='left'),
        cells=dict(values=[df[col] for col in df.columns],
        align='left')
    )]
)
fig.show()

In [28]:
query = """select name, organic_stake/total_stake as share
from 'vdp'
"""

df = pd.read_sql(query, engine)
fig = px.ecdf(df,
              x='share',
              title="Distribution of Organic Stake Share")

for x in [0.05, 0.10, 0.25, 0.50]:
    fig.add_vline(
        x=x,
        line_dash="dash",
        line_color="yellow",
        annotation_text=f"{int(x*100)}%",
        annotation_position="top"
    )
fig.update_layout(
    xaxis_title="Organic Stake Share",
    yaxis_title="Cumulative Share of Validators"
)

fig.update_xaxes(tickformat='.0%')
fig.update_yaxes(tickformat='.0%')

fig.update_traces(hovertemplate=
                  "<b>Organic Stake Share:</b> %{x:.1%}<br>"
                  "<b>Validators Below This Level:</b> %{y:.1%}"
                  "<extra></extra>")
fig.show()